In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, datetime, gc
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Cấu hình đường dẫn
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/candidates/"
MODEL_DIR = BASE_PATH + "outputs/models/"

# 2. Khởi tạo Spark tối ưu cho CPU (Sử dụng tối đa các nhân CPU hiện có)
spark = SparkSession.builder \
    .appName("HM_Final_Train_CPU_Version") \
    .config("spark.driver.memory", "12g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# 3. Xác định mốc thời gian Tuần 7
transactions = spark.read.parquet(INPUT_FILE)
max_date = transactions.select(F.max("t_dat")).collect()[0][0]
test_start_date = max_date - datetime.timedelta(days=7)
val_start_date = test_start_date - datetime.timedelta(days=7)

print(f"✅ Spark Ready (CPU Mode)! Mốc Validation: {val_start_date}")

Mounted at /content/drive
✅ Spark Ready (CPU Mode)! Mốc Validation: 2020-09-07 17:00:00


In [ ]:
print("⏳ Đang chuẩn bị Mapping Article ID...")
articles = spark.read.parquet(BASE_PATH + "processed/articles_processed.parquet")
mapping_7to10 = articles.select("article_id") \
    .withColumn("product_code", F.substring(F.col("article_id"), 1, 7)) \
    .groupBy("product_code").agg(F.first("article_id").alias("article_id_10")).cache()

print(f"✅ Đã chuẩn bị xong mapping cho {mapping_7to10.count():,} sản phẩm.")

⏳ Đang chuẩn bị Mapping Article ID...
✅ Đã chuẩn bị xong mapping cho 47,224 sản phẩm.


In [ ]:
def process_sources():
    def explode_src(file, col):
        return spark.read.parquet(OUTPUT_DIR + file).select("customer_id", F.explode(col).alias("article_id"))

    hist = explode_src("history_candidates_W7.parquet", "history_candidates")
    meta = explode_src("meta_candidates_pro_W7.parquet", "meta_candidates")
    trend = explode_src("trending_candidates_W7.parquet", "trending_candidates")

    fp = spark.read.parquet(OUTPUT_DIR + "fp_association_candidates_val_W7.parquet") \
              .select("customer_id", F.explode("fp_candidates").alias("product_code")) \
              .join(F.broadcast(mapping_7to10), "product_code") \
              .select("customer_id", F.col("article_id_10").alias("article_id"))

    als = spark.read.parquet(OUTPUT_DIR + "als_top100_val_W7_decoded.parquet")
    clip = spark.read.parquet(OUTPUT_DIR + "image_candidates_W7.parquet").repartition(200)

    return hist, meta, trend, fp, als, clip

hist_df, meta_df, trend_df, fp_df, als_df, clip_df = process_sources()
print("✅ Đã nạp xong 6 nguồn ứng viên.")

✅ Đã nạp xong 6 nguồn ứng viên.


In [ ]:
print("🔗 Đang thực hiện Union và Gán nhãn...")
all_union = hist_df.select("customer_id", "article_id") \
    .union(meta_df.select("customer_id", "article_id")) \
    .union(trend_df.select("customer_id", "article_id")) \
    .union(fp_df.select("customer_id", "article_id")) \
    .union(als_df.select("customer_id", "article_id")) \
    .union(clip_df.select("customer_id", "article_id"))

final_candidates = all_union.distinct()

# Tạo nhãn từ giao dịch thực tế Tuần 7
actual_week7 = transactions.filter((F.col("t_dat") >= F.lit(val_start_date)) & (F.col("t_dat") < F.lit(test_start_date))) \
    .select(F.col("customer_id").cast("string"), F.lpad(F.col("article_id").cast("string"), 10, "0").alias("article_id")) \
    .distinct() \
    .withColumn("label", F.lit(1))

train_set = final_candidates.join(actual_week7, ["customer_id", "article_id"], "left").fillna(0)
train_set = train_set.join(als_df, ["customer_id", "article_id"], "left").fillna(0, subset=["als_score"])
train_set = train_set.join(clip_df, ["customer_id", "article_id"], "left").fillna(0, subset=["clip_score"])

print(f"📊 Thống kê: {train_set.count():,} ứng viên đã sẵn sàng.")

🔗 Đang thực hiện Union và Gán nhãn...
📊 Thống kê: 68,773,641 ứng viên đã sẵn sàng.


In [ ]:
print("🛠️ Đang tính toán Feature và Lưu Checkpoint...")
item_feat = transactions.filter(F.col("t_dat") < F.lit(val_start_date)) \
    .groupBy("article_id").agg(F.count("customer_id").alias("item_popularity"), F.avg("price").alias("item_price")) \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0"))

user_feat = transactions.filter(F.col("t_dat") < F.lit(val_start_date)) \
    .groupBy(F.col("customer_id").cast("string")).agg(F.avg("price").alias("user_avg_budget"), F.count("article_id").alias("user_buy_freq"))

final_train_ready = train_set.join(F.broadcast(item_feat), "article_id", "left") \
                             .join(F.broadcast(user_feat), "customer_id", "left").fillna(0)

# Lưu bảng 70 triệu dòng (Checkpoint cực kỳ quan trọng khi chạy CPU)
checkpoint_path = BASE_PATH + "outputs/train_sets/final_train_ready_W7.parquet"
final_train_ready.write.mode("overwrite").parquet(checkpoint_path)
print(f"✅ Đã lưu Checkpoint thành công tại: {checkpoint_path}")

🛠️ Đang tính toán Feature và Lưu Checkpoint...
✅ Đã lưu Checkpoint thành công tại: /content/drive/MyDrive/HM-DATA/outputs/train_sets/final_train_ready_W7.parquet


In [ ]:
import xgboost as xgb

# 1. Lấy mẫu âm 5% để train nhanh trên CPU
pos = final_train_ready.filter(F.col("label") == 1)
neg = final_train_ready.filter(F.col("label") == 0)
train_pd = pos.union(neg.sample(False, 0.05, seed=42)).toPandas()

# 2. Cấu hình Model cho CPU
features = ["als_score", "clip_score", "item_popularity", "item_price", "user_avg_budget", "user_buy_freq"]
X, y = train_pd[features], train_pd["label"]

print("🚀 Đang huấn luyện XGBoost trên CPU...")
model = xgb.XGBClassifier(
    objective="binary:logistic",
    tree_method="hist",  # Histogram vẫn là thuật toán nhanh nhất trên CPU
    n_estimators=300,    # Giảm nhẹ số cây để tăng tốc độ trên CPU
    max_depth=6,
    n_jobs=-1,           # Dùng hết nhân CPU
    random_state=42
)
model.fit(X, y)
model.save_model(BASE_PATH + "outputs/models/xgb_ranker_final.json")
print("✅ Đã lưu mô hình XGBoost (CPU version).")

🚀 Đang huấn luyện XGBoost trên CPU...
✅ Đã lưu mô hình XGBoost (CPU version).


In [ ]:
from pyspark.mllib.evaluation import RankingMetrics

print("🎯 Đang đánh giá MAP@12 tổng lực bằng CPU...")
full_set = spark.read.parquet(BASE_PATH + "outputs/train_sets/final_train_ready_W7.parquet")

def predict_batch_cpu(iterator):
    import xgboost as xgb
    bst = xgb.Booster()
    bst.load_model(BASE_PATH + "outputs/models/xgb_ranker_final.json")
    bst.set_param({'predictor': 'cpu_predictor'}) # Ép dự đoán trên CPU
    for pdf in iterator:
        pdf['pred_score'] = bst.predict(xgb.DMatrix(pdf[features]))
        yield pdf[['customer_id', 'article_id', 'pred_score']]

# Tăng repartition lên 800 để CPU xử lý từng miếng nhỏ hơn
predictions = full_set.repartition(800).mapInPandas(predict_batch_cpu,
    schema="customer_id string, article_id string, pred_score float")

# Xếp hạng Top 12
top_12 = predictions.withColumn("rn", F.row_number().over(Window.partitionBy("customer_id").orderBy(F.desc("pred_score")))) \
    .filter(F.col("rn") <= 12).groupBy("customer_id").agg(F.collect_list("article_id").alias("p"))

# Đáp án thực tế
actuals = actual_week7.groupBy("customer_id").agg(F.collect_list("article_id").alias("a"))

# Tính điểm MAP@12
eval_rdd = top_12.join(actuals, "customer_id", "inner").select("p", "a").rdd.map(lambda r: (list(r[0]), list(r[1])))
print(f"🏆 MAP@12 CUỐI CÙNG (CPU Mode): {RankingMetrics(eval_rdd).meanAveragePrecision:.6f}")

🎯 Đang đánh giá MAP@12 tổng lực bằng CPU...


/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


🏆 MAP@12 CUỐI CÙNG (CPU Mode): 0.020321
